In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils
import features

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    df_cands = "test.parquet"


config = Config()

In [4]:
# %%time

# X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, do_ordinal_encoder_transform=False)
# # X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
# #     config,
# #     X_train,
# #     X_val,
# #     X_val_seq,
# #     ordinal_encoder,
# #     return_only_val=True,
# #     truncate_val=True
# # )
# pad = ordinal_encoder.unknown_value

In [5]:
%%time

PATH_DATA = config.path_data
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')
df_test = df_test.sort(config.col_ts)
df_full = pl.concat([df_train, df_test])

del df_train
gc.collect()

CPU times: user 19.7 s, sys: 7.49 s, total: 27.2 s
Wall time: 10.4 s


3076

In [6]:
df_full = df_full.with_columns(
    [
        pl.col("user").cast(pl.UInt32, strict=False).alias("user"),
        pl.col("aid").cast(pl.UInt32, strict=False).alias("aid"),
    ]
)

df_test = df_test.with_columns(
    [
        pl.col("user").cast(pl.UInt32, strict=False).alias("user"),
        pl.col("aid").cast(pl.UInt32, strict=False).alias("aid"),
    ]
)

In [7]:
# %%time

# PATH_DATA = config.path_data
# df_test = pl.read_parquet(PATH_DATA + "df_test_n_session.parquet")
# df_test = df_test.sort(config.col_ts)

In [8]:
col_user = "session"
col_item = "item_id"

In [9]:
%%time

df_full = df_full.to_pandas()
df_full = utils.make_date_stats(df_full)
df_full = df_full.drop('date_day', axis=1)
df_full = pl.from_pandas(df_full)

CPU times: user 1min 45s, sys: 13.9 s, total: 1min 59s
Wall time: 1min 56s


In [10]:
%%time

df_test = df_test.to_pandas()
df_test = utils.make_date_stats(df_test)
df_test = df_test.drop('date_day', axis=1)
df_test = pl.from_pandas(df_test)

CPU times: user 3.3 s, sys: 365 ms, total: 3.66 s
Wall time: 3.55 s


In [11]:
df_test.head()

user,aid,ts,type,n_session,date,n_day,n_week,dayofweek,hour
u32,u32,i64,str,i64,datetime[ns],i8,i8,i8,i8
12899779,59625,1661724000278,"""clicks""",0,2022-08-29 00:00:00.278000128,29,35,0,0
12899780,1142000,1661724000378,"""clicks""",0,2022-08-29 00:00:00.377999872,29,35,0,0
12899781,141736,1661724000559,"""clicks""",0,2022-08-29 00:00:00.559000064,29,35,0,0
12899782,1669402,1661724000568,"""clicks""",0,2022-08-29 00:00:00.568,29,35,0,0
12899783,255297,1661724000572,"""clicks""",0,2022-08-29 00:00:00.572,29,35,0,0


In [12]:
df_test = df_test.rename({"user": "session"})
df_test = df_test.rename({"aid": "item_id"})

df_full = df_full.rename({"user": "session"})
df_full = df_full.rename({"aid": "item_id"})

In [13]:
df_test.head()

session,item_id,ts,type,n_session,date,n_day,n_week,dayofweek,hour
u32,u32,i64,str,i64,datetime[ns],i8,i8,i8,i8
12899779,59625,1661724000278,"""clicks""",0,2022-08-29 00:00:00.278000128,29,35,0,0
12899780,1142000,1661724000378,"""clicks""",0,2022-08-29 00:00:00.377999872,29,35,0,0
12899781,141736,1661724000559,"""clicks""",0,2022-08-29 00:00:00.559000064,29,35,0,0
12899782,1669402,1661724000568,"""clicks""",0,2022-08-29 00:00:00.568,29,35,0,0
12899783,255297,1661724000572,"""clicks""",0,2022-08-29 00:00:00.572,29,35,0,0


In [14]:
%%time

dict_df_history_cnt_interactions_types = features.get_user_item_stats(df_test)

CPU times: user 13.5 s, sys: 3.01 s, total: 16.5 s
Wall time: 928 ms


In [15]:
%%time

item_popularity_stats = features.get_item_popularity_stats(df_full, col_item=col_item)
dict_user_item_popularity_stats = features.get_user_item_popularity_stats(df_test, item_popularity_stats)

CPU times: user 2min 7s, sys: 28 s, total: 2min 35s
Wall time: 12.3 s


In [16]:
# del df_full
# gc.collect()

In [17]:
%%time

for i in tqdm(range(17)):
    path = f'{config.cur_model_output_path}test_{i}.parquet'
    df = pl.read_parquet(path)
    df = df.with_columns(
        [
            pl.col("session").cast(pl.UInt32, strict=False).alias("session"),
            pl.col("item_id").cast(pl.UInt32, strict=False).alias("item_id"),
        ]
    )

    df = features.make_rank_features(df)
    df = features.make_item_interaction_to_interaction_features(df, df_full)
    df = features.make_item_last_interation_features(df, df_test)
    df = features.make_user_item_features(df, dict_df_history_cnt_interactions_types)
    df = features.make_user_item_popularity_features(df, item_popularity_stats, dict_user_item_popularity_stats)
    df = features.make_user_features(df, df_test)
    df = features.make_inverse_rank_features(df)

    cols_rank = [col for col in df.columns if col.endswith('_rank')]
    cols_rank.extend(['rank_min', 'rank_max', 'rank_cnt_nulls'])
    for col in cols_rank:
        df = df.with_column(pl.col(col).fill_null(-99))
        df = df.with_column(
            pl.col(col).cast(pl.Int8, strict=False).alias(col)
        )

    df = df.sort('session')
    final_name = f'{config.cur_model_output_path}test_w_features_{i}.parquet'
    df.write_parquet(final_name)

  0%|          | 0/17 [00:00<?, ?it/s]

CPU times: user 3h 11min, sys: 28min 17s, total: 3h 39min 17s
Wall time: 28min 3s


In [18]:
df.head()

item_id,session,rnn_score,rnn_rank,weak_nns_score,weak_nns_rank,cooc_score,cooc_rank,bm25_score,bm25_rank,tfidf_score,tfidf_rank,cosine_score,cosine_rank,last_items_rank,rank_avg,rank_min,rank_max,rank_std,rank_median,rank_cnt_nulls,ratio_clicks_to_carts,ratio_clicks_to_orders,ratio_carts_to_orders,ratio_clicks_to_carts_test_week,ratio_clicks_to_orders_test_week,ratio_carts_to_orders_test_week,time_since_last_clicks_this_item_by_user,time_since_last_carts_this_item_by_user,time_since_last_orders_this_item_by_user,time_since_last_interaction_this_item_by_user,last_interaction_type,cnt_interactions_total,cnt_interactions_clicks,cnt_interactions_carts,cnt_interactions_orders,ratio_user_clicks_to_carts,ratio_user_clicks_to_orders,ratio_user_carts_to_orders,item_popularity,item_popularity_total_mean,item_popularity_total_median,item_popularity_total_std,item_popularity_total_min,item_popularity_total_max,item_popularity_to_item_popularity_total_mean,item_popularity_to_item_popularity_total_median,item_popularity_clicks_mean,item_popularity_clicks_median,item_popularity_clicks_std,item_popularity_clicks_min,item_popularity_clicks_max,item_popularity_to_item_popularity_clicks_mean,item_popularity_to_item_popularity_clicks_median,item_popularity_carts_mean,item_popularity_carts_median,item_popularity_carts_std,item_popularity_carts_min,item_popularity_carts_max,item_popularity_to_item_popularity_carts_mean,item_popularity_to_item_popularity_carts_median,item_popularity_orders_mean,item_popularity_orders_median,item_popularity_orders_std,item_popularity_orders_min,item_popularity_orders_max,item_popularity_to_item_popularity_orders_mean,item_popularity_to_item_popularity_orders_median,ratio_user_total_clicks_to_carts,ratio_user_total_clicks_to_orders,ratio_user_total_carts_to_orders,inverse_rank_level_1
u32,u32,f32,i8,i32,i8,f32,i8,f32,i8,f32,i8,f32,i8,i8,f32,i8,i8,f32,f32,i8,f32,f32,f32,f32,f32,f32,u32,u32,u32,u32,u8,u8,u8,u8,u8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
792934,14499779,0.089881,1,null,-99,0.502259,3,2393.769775,1,0.446578,1,0.446578,1,1,1.333333,1,3,0.816497,1.0,1,0.084529,0.028522,0.337423,0.109211,0.014474,0.13253,0,null,null,0,0,1,1,0,0,0.0,0.0,NaN,50.47073,74.561348,27.314634,102.904854,3.241463,336.278046,0.676902,1.847754,74.561348,27.314634,102.904854,3.241463,336.278046,0.676902,1.847754,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,5.333333
986164,14499779,0.045398,2,1,33,0.70947,2,750.374329,4,0.355017,2,0.355017,2,2,6.714286,2,33,11.614851,2.0,0,0.160891,0.058106,0.361155,0.149902,0.02544,0.169713,787854,null,null,787854,0,2,2,0,0,0.0,0.0,NaN,198.91951,74.561348,27.314634,102.904854,3.241463,336.278046,2.667864,7.282525,74.561348,27.314634,102.904854,3.241463,336.278046,2.667864,7.282525,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2.780303
1782099,14499779,0.031609,3,null,-99,null,-99,311.612823,22,0.082832,8,0.072799,9,-99,10.5,3,22,8.103498,8.5,3,0.083436,0.024214,0.290214,0.060543,0.012526,0.206897,null,null,null,null,null,0,0,0,0,NaN,NaN,NaN,82.892685,74.561348,27.314634,102.904854,3.241463,336.278046,1.111738,3.034735,74.561348,27.314634,102.904854,3.241463,336.278046,1.111738,3.034735,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.614899
1120524,14499779,0.0203,4,null,-99,null,-99,210.142105,54,0.059563,15,0.047437,18,-99,22.75,4,54,21.685249,16.5,3,0.078294,0.022321,0.285091,0.050505,0.014141,0.28,null,null,null,null,null,0,0,0,0,NaN,NaN,NaN,80.407318,74.561348,27.314634,102.904854,3.241463,336.278046,1.078405,2.943745,74.561348,27.314634,102.904854,3.241463,336.278046,1.078405,2.943745,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.390741
1078347,14499779,0.019107,5,null,-99,0.107998,46,505.722778,6,0.098502,5,0.082148,7,-99,13.8,5,46,18.0194

In [19]:
stop

NameError: name 'stop' is not defined

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
%%time

df = features.make_rank_features(df)

In [ ]:
%%time

df = features.make_user_item_features(df, df_test)

In [ ]:
col_user = "session"
col_item = "item_id"

In [ ]:
df

In [ ]:
df = df.sort(col_user)

In [ ]:
%%time

df.write_parquet(config.cur_model_output_path + 'train_clicks_orders_w_features.parquet')

In [ ]:
stop

In [ ]:
X_val.shape

In [ ]:
X_val = X_val.filter(pl.col("type") != "clicks")

In [ ]:
X_val.shape

In [ ]:
X_val

In [ ]:
mod = "val"

In [ ]:
%%time

dict_models_preds = dict()

cols_to_drop = ['index']

cols_to_rename = dict()
cols_to_rename[config.col_user] = 'session'
cols_to_rename['item_id'] = 'preds'

cols_to_add_model_name = ['preds']

model_names = ['rnn', 'cooc_smart', 'bm25', 'tfidf', 'cosine', 'last_items']

for model_name in model_names:
    dict_models_preds[model_name] = pl.read_parquet(config.cur_model_output_path + f'cur_{model_name}_df_{mod}_preds_100.parquet')
    for col in cols_to_drop:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].drop(col)
    for col in cols_to_rename:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: cols_to_rename[col]})
    for col in cols_to_add_model_name:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: model_name + '_' + col})
    if model_name == 'rnn':
        dict_models_preds[model_name] = dict_models_preds[model_name].rename({'preds_scores': 'rnn_scores'})
        dict_models_preds[model_name] = dict_models_preds[model_name].with_column(
            pl.col("rnn_scores")
                .rank(method="ordinal", reverse=True)
                .over(["session"])
                .alias("rnn_rank")
        )
    elif model_name in ['cooc_smart']:
        for col in ['cooc_triplet_score_0', 'cooc_triplet_rank_0', 'cooc_score_0', 'cooc_rank_0']:
            if col in dict_models_preds[model_name]:
                dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: col[:-2]})
    for col in dict_models_preds[model_name].columns:
        if 'preds' in col:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: 'item_id'})

In [ ]:
dict_models_preds["bm25"].head()

In [ ]:
dict_models_preds["last_items"].head()

In [ ]:
model_names

In [ ]:
%%time

df_merged = dict_models_preds[model_names[0]]
for i in tqdm(range(1, len(model_names))):
    model_name_cur = model_names[i]
    df_merged = utils.polars_outer_merge(df_merged, dict_models_preds[model_name_cur])

In [ ]:
%%time

X_val_lists = X_val.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

In [ ]:
%%time

df_targets = utils.create_targets_df(list(X_val_lists.to_pandas()[config.col_train_seq]))

In [ ]:
%%time

df_users = pl.from_pandas(X_val_lists.to_pandas()[[config.col_user]].reset_index())
df_users = df_users.join(df_targets, on='index', how='inner')
df_users = df_users.drop('index')
df_users = df_users.rename({config.col_user: 'session'})

In [ ]:
%%time

df_merged = utils.polars_outer_merge(df_merged, df_users)

In [ ]:
del df_users, dict_models_preds
gc.collect()

In [ ]:
columns_from_models = [
    col for col in df_merged.columns if "_score" in col or "_rank" in col
]

In [ ]:
# %%time

# df_merged = pl.from_pandas(df_merged.to_pandas().dropna(subset=columns_from_models, thresh=1))

In [ ]:
%%time

df_merged = df_merged.with_column(pl.col("target").fill_null(0))
df_merged = df_merged.with_columns(
        [
            pl.col('target')
            .cast(pl.Float32)
            .alias('target'),
        ]
    )

In [ ]:
df_merged.head()

In [ ]:
df_merged.shape

In [ ]:
df_merged = df_merged.sort("session")

In [ ]:
df_merged.head()

In [ ]:
neg_sample_size = len(df_merged.filter(pl.col("target") > 0)) * 20
df_merged = pl.concat(
    [
        df_merged.filter(pl.col("target") > 0),
        df_merged.filter(pl.col("target") == 0).sample(neg_sample_size, seed=0),
    ]
)

In [ ]:
df_merged = df_merged.sort("session")

In [ ]:
%%time

df_merged.write_parquet(config.cur_model_output_path + 'train_clicks_orders.parquet')

In [ ]:
stop

In [ ]:
qids_train = df_merged.to_pandas().groupby(["session"])["item_id"].count().values

In [ ]:
qids_train

In [ ]:
from lightgbm import LGBMRanker

In [ ]:
df_merged

In [ ]:
def find_query(q: str, data: list) -> list:
    path_list = list()

    def helper(subdata, path):
        for index, item in enumerate(subdata):
            if isinstance(item, list):
                helper(item, path + [index])
            elif item == q:
                path_list.append(path + [index])

    helper(data, list())
    return path_list

In [ ]:
data = ["a", ["b", "c", ["d", "a"]]]

In [ ]:
find_query("d", data)

In [ ]:
import lightgbm as lgb

In [ ]:
bst = lgb.Booster(model_file="../src/otto_recommender/booster_fold2.lgb")

In [ ]:
importance_df = (
    pd.DataFrame(
        {
            "feature_name": bst.feature_name(),
            "importance_gain": bst.feature_importance(importance_type="gain"),
            "importance_split": bst.feature_importance(importance_type="split"),
        }
    )
    .sort_values("importance_gain", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
importance_df

In [ ]:
lgb.plot_importance(bst, importance_type="gain")

In [ ]:
df_merged = pl.read_parquet(
    config.cur_model_output_path + "train_clicks_orders.parquet"
).to_pandas()

In [ ]:
df_merged["session"].nunique()

In [ ]:
df_merged[df_merged["session"] == 13405261]

In [ ]:
df_merged

In [ ]:
%%time

columns = [
    col for col in df_merged.columns if "_score" in col or "_rank" in col
]
X_train = df_merged[columns].to_numpy(dtype=np.float32)
df_merged['target'] = (df_merged['target'] > 0) * 1
y_train = df_merged['target'].astype(np.int8)

In [ ]:
%%time

train_groups = utils.get_group_for_lgb(df_merged["session"].values)

In [ ]:
%%time

lgb_train = lgb.Dataset(
    X_train,
    y_train,
    feature_name=columns,
    group=train_groups,
    free_raw_data=False,
)

In [ ]:
%%time

lgb_train.construct()

In [ ]:
lgb_train.get_label()

In [ ]:
lgb_train.get_feature_name()

In [ ]:
len(lgb_train.get_data())

In [ ]:
%%time

len(np.where((np.isnan(lgb_train.get_data()).all(axis=1) == False) & (lgb_train.get_label() == 1))[0])

In [ ]:
df_merged.loc[
    np.where(
        (np.isnan(lgb_train.get_data()).all(axis=1) == False)
        & (lgb_train.get_label() == 1)
    )[0]
]

In [ ]:
946438 / len(df_merged[df_merged["target"] == 1])

In [ ]:
df_merged.loc[
    np.where(
        (np.isnan(lgb_train.get_data()).all(axis=1) == True)
        & (lgb_train.get_label() == 0)
    )[0]
]